# Notebook 02 — STDP Learning Window Demo

Demonstrates the classical pair STDP rule by:
1. Constructing paired pre/post spike trains with controlled Δt offsets.
2. Running `PairSTDP.forward()` and recording the resulting ΔW.
3. Plotting the STDP learning window (ΔW vs. Δt).

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt

from plasticity import PairSTDP

In [ ]:
n = 1       # single synapse
T = 200     # timesteps
dt_offsets = np.arange(-50, 51, 5)  # pre-post timing differences in ms

stdp = PairSTDP(n_pre=n, n_post=n, A_plus=0.01, A_minus=0.012, dt=1.0)
initial_w = stdp.weight.data.clone()
dW_list = []

for offset in dt_offsets:
    # Reset weights to initial value
    stdp.weight.data.copy_(initial_w)
    trace_pre, trace_post = stdp.init_traces(batch=1, device=torch.device('cpu'))

    pre_spike_t  = T // 2
    post_spike_t = pre_spike_t + int(offset)

    for t in range(T):
        pre  = torch.tensor([[1.0]]) if t == pre_spike_t  else torch.zeros(1, n)
        post = torch.tensor([[1.0]]) if t == post_spike_t else torch.zeros(1, n)
        dw, trace_pre, trace_post = stdp(pre, post, trace_pre, trace_post)

    dW_list.append((stdp.weight.data - initial_w).item())

plt.figure(figsize=(7, 4))
plt.axhline(0, color='gray', lw=0.8)
plt.axvline(0, color='gray', lw=0.8)
plt.plot(dt_offsets, dW_list, 'o-', color='steelblue')
plt.xlabel('Δt = t_post − t_pre (ms)')
plt.ylabel('ΔW')
plt.title('STDP Learning Window')
plt.tight_layout()
plt.show()